# Greedy Experiments

Write your experiments in this document. The experiments below explore greedy Gray code constructions for **integer partitions** of n.

# Integer Partition Experiments

This section explores greedy Gray code constructions for integer partitions of n. We try two types of operations:
- **Move-one-unit**: move 1 unit from one part to another (can create or destroy parts of size 1)
- **Split/merge**: split one part into two, or merge two parts into one

For each operation type we experiment with:
- Different **starting partitions**: `(n,)` (single large part) vs `(1,1,...,1)` (all ones)
- Different **priority rules**: lexicographically smallest vs lexicographically largest next partition

Key findings:
- Move-one-unit with lex-min priority starting from `(1,...,1)` works for small n but fails at n=6.
- **Split/merge with lex-min priority starting from `(1,...,1)` works for n=1 through n=7** but fails at n=8 (missing only the partition `(7,1)`).
- Split/merge with fewest-parts priority starting from `(1,...,1)` also gets very close (misses by 1 for n=8,9,10,11).

## Objects: Integer Partitions

In [2]:
# The set of all integer partitions of n.
# Each partition is represented as a tuple in weakly decreasing order.
# For example, the partitions of 4 are: (4,), (3,1), (2,2), (2,1,1), (1,1,1,1).
def allPartitions(n):
  S = set()
  def helper(remaining, maxPart, current):
    if remaining == 0:
      S.add(tuple(current))
      return
    for part in range(min(remaining, maxPart), 0, -1):
      helper(remaining - part, part, current + [part])
  helper(n, n, [])
  return S

In [3]:
# Test the set of all objects.
n = 10
S = allPartitions(n)
for p in sorted(S):
  print(p)
print("total: %d" % len(S))

(1, 1, 1, 1, 1, 1, 1, 1, 1, 1)
(2, 1, 1, 1, 1, 1, 1, 1, 1)
(2, 2, 1, 1, 1, 1, 1, 1)
(2, 2, 2, 1, 1, 1, 1)
(2, 2, 2, 2, 1, 1)
(2, 2, 2, 2, 2)
(3, 1, 1, 1, 1, 1, 1, 1)
(3, 2, 1, 1, 1, 1, 1)
(3, 2, 2, 1, 1, 1)
(3, 2, 2, 2, 1)
(3, 3, 1, 1, 1, 1)
(3, 3, 2, 1, 1)
(3, 3, 2, 2)
(3, 3, 3, 1)
(4, 1, 1, 1, 1, 1, 1)
(4, 2, 1, 1, 1, 1)
(4, 2, 2, 1, 1)
(4, 2, 2, 2)
(4, 3, 1, 1, 1)
(4, 3, 2, 1)
(4, 3, 3)
(4, 4, 1, 1)
(4, 4, 2)
(5, 1, 1, 1, 1, 1)
(5, 2, 1, 1, 1)
(5, 2, 2, 1)
(5, 3, 1, 1)
(5, 3, 2)
(5, 4, 1)
(5, 5)
(6, 1, 1, 1, 1)
(6, 2, 1, 1)
(6, 2, 2)
(6, 3, 1)
(6, 4)
(7, 1, 1, 1)
(7, 2, 1)
(7, 3)
(8, 1, 1)
(8, 2)
(9, 1)
(10,)
total: 42


## Operations: Move-One-Unit and Split/Merge

In [4]:
# Move-one-unit: move 1 unit from one part to another.
# If a part of size 1 loses its unit, that part is removed.
# A unit can also move into a brand-new part of size 1.
def neighborsMove(partition):
  parts = list(partition)
  results = set()
  n = sum(parts)
  for i in range(len(parts)):
    for j in range(len(parts)):
      if i == j:
        continue
      new_parts = parts[:]
      new_parts[i] -= 1
      new_parts[j] += 1
      new_parts = [p for p in new_parts if p > 0]
      new_parts_sorted = tuple(sorted(new_parts, reverse=True))
      if new_parts_sorted and sum(new_parts_sorted) == n:
        results.add(new_parts_sorted)
    # Split off a new part of size 1 from part i
    if parts[i] > 1:
      new_parts = parts[:]
      new_parts[i] -= 1
      new_parts.append(1)
      results.add(tuple(sorted(new_parts, reverse=True)))
  return results - {partition}

# Split/merge: split one part into two positive parts, or merge two parts into one.
def neighborsSplitMerge(partition):
  parts = list(partition)
  results = set()
  # Splits
  for i in range(len(parts)):
    for a in range(1, parts[i]):
      b = parts[i] - a
      new_parts = parts[:i] + parts[i+1:] + [a, b]
      results.add(tuple(sorted(new_parts, reverse=True)))
  # Merges
  for i in range(len(parts)):
    for j in range(i+1, len(parts)):
      merged = parts[i] + parts[j]
      new_parts = [parts[k] for k in range(len(parts)) if k != i and k != j] + [merged]
      results.add(tuple(sorted(new_parts, reverse=True)))
  return results - {partition}

In [5]:
# Testing the operations
print("Move-one-unit neighbors of (3, 2, 1):")
for p in sorted(neighborsMove((3, 2, 1))):
  print(" ", p)

print("\nSplit/merge neighbors of (3, 2, 1):")
for p in sorted(neighborsSplitMerge((3, 2, 1))):
  print(" ", p)

Move-one-unit neighbors of (3, 2, 1):
  (2, 2, 1, 1)
  (2, 2, 2)
  (3, 1, 1, 1)
  (3, 3)
  (4, 1, 1)
  (4, 2)

Split/merge neighbors of (3, 2, 1):
  (2, 2, 1, 1)
  (3, 1, 1, 1)
  (3, 3)
  (4, 2)
  (5, 1)


## Experiments

In [6]:
# Generic greedy algorithm for integer partitions.
# Inputs: n (integer to partition), start (starting partition),
#         neighborsFunc (function returning neighbors),
#         priority_key (key function for choosing next partition).
def greedyPartition(n, start, neighborsFunc, priority_key):
  visited = set()
  current = start
  visited.add(current)
  yield current
  while True:
    candidates = [c for c in neighborsFunc(current) if c not in visited]
    if not candidates:
      break
    current = min(candidates, key=priority_key)
    visited.add(current)
    yield current

In [7]:
# --- MOVE-ONE-UNIT EXPERIMENTS ---

# Experiment 0: Move-one-unit, start=(n,), priority=lex min
def greedyMoveLexMin_startN(n):
  yield from greedyPartition(n, (n,), neighborsMove, lambda p: p)

# Experiment 1: Move-one-unit, start=(n,), priority=lex max
def greedyMoveLexMax_startN(n):
  yield from greedyPartition(n, (n,), neighborsMove, lambda p: tuple(-x for x in p))

# Experiment 2: Move-one-unit, start=(1,...,1), priority=lex min
def greedyMoveLexMin_startOnes(n):
  yield from greedyPartition(n, (1,)*n, neighborsMove, lambda p: p)

# Experiment 3: Move-one-unit, start=(1,...,1), priority=lex max
def greedyMoveLexMax_startOnes(n):
  yield from greedyPartition(n, (1,)*n, neighborsMove, lambda p: tuple(-x for x in p))

In [8]:
# --- SPLIT/MERGE EXPERIMENTS ---

# Experiment 4: Split/merge, start=(n,), priority=lex min
def greedySMLexMin_startN(n):
  yield from greedyPartition(n, (n,), neighborsSplitMerge, lambda p: p)

# Experiment 5: Split/merge, start=(n,), priority=lex max
def greedySMLexMax_startN(n):
  yield from greedyPartition(n, (n,), neighborsSplitMerge, lambda p: tuple(-x for x in p))

# Experiment 6: Split/merge, start=(1,...,1), priority=lex min
# This is the most promising experiment!
def greedySMLexMin_startOnes(n):
  yield from greedyPartition(n, (1,)*n, neighborsSplitMerge, lambda p: p)

# Experiment 7: Split/merge, start=(1,...,1), priority=lex max
def greedySMLexMax_startOnes(n):
  yield from greedyPartition(n, (1,)*n, neighborsSplitMerge, lambda p: tuple(-x for x in p))

# Experiment 8: Split/merge, start=(1,...,1), priority=fewest parts (then lex min)
def greedySM_fewestParts_startOnes(n):
  yield from greedyPartition(n, (1,)*n, neighborsSplitMerge, lambda p: (len(p), p))

# Experiment 9: Split/merge, start=(1,...,1), priority=most parts (then lex min)
def greedySM_mostParts_startOnes(n):
  yield from greedyPartition(n, (1,)*n, neighborsSplitMerge, lambda p: (-len(p), p))

## Results

In [9]:
# Run all experiments for n between nMin and nMax.
experiments = [
  greedyMoveLexMin_startN,
  greedyMoveLexMax_startN,
  greedyMoveLexMin_startOnes,
  greedyMoveLexMax_startOnes,
  greedySMLexMin_startN,
  greedySMLexMax_startN,
  greedySMLexMin_startOnes,
  greedySMLexMax_startOnes,
  greedySM_fewestParts_startOnes,
  greedySM_mostParts_startOnes,
]
nMin = 1
nMax = 10

for experimentNum, experiment in enumerate(experiments):
  print("\n\n" + "Experiment %d: " % experimentNum + experiment.__name__)

  results = []
  for n in range(nMin, nMax+1):
    print("\nn = %d" % n)
    allObjects = allPartitions(n)
    goal = len(allObjects)
    objects = []
    for partition in experiment(n):
      print(partition)
      objects.append(partition)
    numObjects = len(objects)
    print("total: %d / %d" % (numObjects, goal))
    result = (set(objects) == set(allObjects))
    results.append(result)

  print("\nSummary of " + experiment.__name__ + ":")
  numExperiments = nMax - nMin + 1
  for i in range(numExperiments):
    n = i + nMin
    print("n = %d: %s" % (n, results[i]))



Experiment 0: greedyMoveLexMin_startN

n = 1
(1,)
total: 1 / 1

n = 2
(2,)
(1, 1)
total: 2 / 2

n = 3
(3,)
(2, 1)
(1, 1, 1)
total: 3 / 3

n = 4
(4,)
(3, 1)
(2, 1, 1)
(1, 1, 1, 1)
total: 4 / 5

n = 5
(5,)
(4, 1)
(3, 1, 1)
(2, 1, 1, 1)
(1, 1, 1, 1, 1)
total: 5 / 7

n = 6
(6,)
(5, 1)
(4, 1, 1)
(3, 1, 1, 1)
(2, 1, 1, 1, 1)
(1, 1, 1, 1, 1, 1)
total: 6 / 11

n = 7
(7,)
(6, 1)
(5, 1, 1)
(4, 1, 1, 1)
(3, 1, 1, 1, 1)
(2, 1, 1, 1, 1, 1)
(1, 1, 1, 1, 1, 1, 1)
total: 7 / 15

n = 8
(8,)
(7, 1)
(6, 1, 1)
(5, 1, 1, 1)
(4, 1, 1, 1, 1)
(3, 1, 1, 1, 1, 1)
(2, 1, 1, 1, 1, 1, 1)
(1, 1, 1, 1, 1, 1, 1, 1)
total: 8 / 22

n = 9
(9,)
(8, 1)
(7, 1, 1)
(6, 1, 1, 1)
(5, 1, 1, 1, 1)
(4, 1, 1, 1, 1, 1)
(3, 1, 1, 1, 1, 1, 1)
(2, 1, 1, 1, 1, 1, 1, 1)
(1, 1, 1, 1, 1, 1, 1, 1, 1)
total: 9 / 30

n = 10
(10,)
(9, 1)
(8, 1, 1)
(7, 1, 1, 1)
(6, 1, 1, 1, 1)
(5, 1, 1, 1, 1, 1)
(4, 1, 1, 1, 1, 1, 1)
(3, 1, 1, 1, 1, 1, 1, 1)
(2, 1, 1, 1, 1, 1, 1, 1, 1)
(1, 1, 1, 1, 1, 1, 1, 1, 1, 1)
total: 10 / 42

Summary of greedyMoveLexMi

# Further Investigation of Split/Merge Experiments (4–9)

This section investigates each split/merge experiment in more detail:
- Full listings for small n
- For failing cases: which partitions are missed and why
- Root-cause analysis of the structural traps that cause failure

**Key findings from this investigation:**
- **Experiment 4** (lex-min from `(n,)`) fails at n=4: misses `(3,1)` because lex-min immediately splits `(4,)` into `(2,2)`, cutting off the path to `(3,1)`.
- **Experiment 5** (lex-max from `(n,)`) works for n=1,2,3,5,7 but fails at n=4,6,8,9,...
- **Experiment 6** (lex-min from `(1,...,1)`) is the strongest: works for n=1–7, fails at n=8 missing only `(7,1)`. Root cause: at `(6,1,1)` the algorithm chose `(6,2)` over `(7,1)` because `(6,2) < (7,1)` lexicographically, stranding `(7,1)`.
- **Experiment 7** (lex-max from `(1,...,1)`) fails earlier at n=6, missing the small partitions `(2,2,1,1)` and `(2,2,2)`.
- **Experiments 8 and 9** (fewest/most-parts from `(1,...,1)`) also fail at n=8, each missing a single partition.
- All experiments starting from `(n,)` perform worse than those starting from `(1,...,1)`.


## Failure Analysis Helper

In [10]:
# Helper: run an experiment and report what was missed and where it got stuck.
def analyzeExperiment(name, genFunc, nMin=1, nMax=10, printListings=True, listingMax=7):
  print("\n" + "="*55)
  print(f"Experiment: {name}")
  print("="*55)

  for n in range(nMin, nMax+1):
    allObjs = allPartitions(n)
    goal = len(allObjs)
    objects = list(genFunc(n))
    success = set(objects) == allObjs
    label = "OK" if success else "FAIL"

    if printListings and n <= listingMax:
      print(f"\nn={n} ({label}):")
      for p in objects:
        print(" ", p)
      if not success:
        missed = sorted(allObjs - set(objects))
        print(f"  MISSED: {missed}")
    else:
      missed = sorted(allObjs - set(objects))
      stuck = objects[-1]
      stuck_nbrs = sorted(neighborsSplitMerge(stuck))
      print(f"\nn={n} ({label}): visited {len(objects)}/{goal}")
      if not success:
        print(f"  Missed:        {missed}")
        print(f"  Last visited:  {stuck}")
        print(f"  Its neighbors: {stuck_nbrs}")


## Experiment 4: Split/Merge, Lex-Min, Start = (n,)

In [11]:
# Experiment 4: Split/merge, start=(n,), priority=lex min.
# Fails at n=4. Starting from (n,), lex-min immediately splits into equal halves,
# cutting off partitions like (3,1) that require passing through (n,) again.
analyzeExperiment(
  "Exp 4: SM LexMin start=(n,)",
  lambda n: greedyPartition(n, (n,), neighborsSplitMerge, lambda p: p),
  nMax=10, listingMax=5
)


Experiment: Exp 4: SM LexMin start=(n,)

n=1 (OK):
  (1,)

n=2 (OK):
  (2,)
  (1, 1)

n=3 (OK):
  (3,)
  (2, 1)
  (1, 1, 1)

n=4 (FAIL):
  (4,)
  (2, 2)
  (2, 1, 1)
  (1, 1, 1, 1)
  MISSED: [(3, 1)]

n=5 (FAIL):
  (5,)
  (3, 2)
  (2, 2, 1)
  (2, 1, 1, 1)
  (1, 1, 1, 1, 1)
  MISSED: [(3, 1, 1), (4, 1)]

n=6 (FAIL): visited 6/11
  Missed:        [(2, 2, 2), (3, 1, 1, 1), (4, 1, 1), (4, 2), (5, 1)]
  Last visited:  (1, 1, 1, 1, 1, 1)
  Its neighbors: [(2, 1, 1, 1, 1)]

n=7 (FAIL): visited 7/15
  Missed:        [(3, 1, 1, 1, 1), (3, 2, 1, 1), (3, 3, 1), (4, 1, 1, 1), (4, 2, 1), (5, 1, 1), (5, 2), (6, 1)]
  Last visited:  (1, 1, 1, 1, 1, 1, 1)
  Its neighbors: [(2, 1, 1, 1, 1, 1)]

n=8 (FAIL): visited 8/22
  Missed:        [(3, 1, 1, 1, 1, 1), (3, 2, 1, 1, 1), (3, 2, 2, 1), (3, 3, 1, 1), (3, 3, 2), (4, 1, 1, 1, 1), (4, 2, 1, 1), (4, 3, 1), (5, 1, 1, 1), (5, 2, 1), (5, 3), (6, 1, 1), (6, 2), (7, 1)]
  Last visited:  (1, 1, 1, 1, 1, 1, 1, 1)
  Its neighbors: [(2, 1, 1, 1, 1, 1, 1)]

n=9 (FAI

## Experiment 5: Split/Merge, Lex-Max, Start = (n,)

In [12]:
# Experiment 5: Split/merge, start=(n,), priority=lex max.
# Works for n=1,2,3,5,7 but fails at n=4,6,8+.
# Lex-max greedily picks the partition with the largest first part,
# which navigates well for some n but creates structural traps for others.
analyzeExperiment(
  "Exp 5: SM LexMax start=(n,)",
  lambda n: greedyPartition(n, (n,), neighborsSplitMerge, lambda p: tuple(-x for x in p)),
  nMax=10, listingMax=7
)


Experiment: Exp 5: SM LexMax start=(n,)

n=1 (OK):
  (1,)

n=2 (OK):
  (2,)
  (1, 1)

n=3 (OK):
  (3,)
  (2, 1)
  (1, 1, 1)

n=4 (FAIL):
  (4,)
  (3, 1)
  (2, 1, 1)
  (2, 2)
  MISSED: [(1, 1, 1, 1)]

n=5 (OK):
  (5,)
  (4, 1)
  (3, 1, 1)
  (3, 2)
  (2, 2, 1)
  (2, 1, 1, 1)
  (1, 1, 1, 1, 1)

n=6 (FAIL):
  (6,)
  (5, 1)
  (4, 1, 1)
  (4, 2)
  (3, 2, 1)
  (3, 3)
  MISSED: [(1, 1, 1, 1, 1, 1), (2, 1, 1, 1, 1), (2, 2, 1, 1), (2, 2, 2), (3, 1, 1, 1)]

n=7 (OK):
  (7,)
  (6, 1)
  (5, 1, 1)
  (5, 2)
  (4, 2, 1)
  (4, 3)
  (3, 3, 1)
  (3, 2, 1, 1)
  (3, 2, 2)
  (2, 2, 2, 1)
  (2, 2, 1, 1, 1)
  (4, 1, 1, 1)
  (3, 1, 1, 1, 1)
  (2, 1, 1, 1, 1, 1)
  (1, 1, 1, 1, 1, 1, 1)

n=8 (FAIL): visited 18/22
  Missed:        [(1, 1, 1, 1, 1, 1, 1, 1), (2, 1, 1, 1, 1, 1, 1), (2, 2, 1, 1, 1, 1), (3, 1, 1, 1, 1, 1)]
  Last visited:  (2, 2, 2, 2)
  Its neighbors: [(2, 2, 2, 1, 1), (4, 2, 2)]

n=9 (FAIL): visited 19/30
  Missed:        [(1, 1, 1, 1, 1, 1, 1, 1, 1), (2, 1, 1, 1, 1, 1, 1, 1), (2, 2, 1, 1, 1, 1, 1

## Experiment 6: Split/Merge, Lex-Min, Start = (1,...,1)  ← Most Promising

In [13]:
# Experiment 6: Split/merge, start=(1,...,1), priority=lex min.
# The strongest experiment: works for n=1 through n=7.
# Fails at n=8, missing only (7,1).
#
# Root cause at n=8: at partition (6,1,1), both (6,2) and (7,1) are unvisited.
# Lex-min chose (6,2) because (6,2) < (7,1). After visiting (6,2), (8,), and (4,4),
# all three neighbors of (7,1) were already visited, stranding it.
analyzeExperiment(
  "Exp 6: SM LexMin start=(1,...,1)",
  lambda n: greedyPartition(n, (1,)*n, neighborsSplitMerge, lambda p: p),
  nMax=12, listingMax=7
)

# Show the exact decision point at n=8
print("\n--- Root cause trace at n=8 ---")
print("Neighbors of (7,1):", sorted(neighborsSplitMerge((7,1))))
print("At (6,1,1), unvisited candidates are (6,2) and (7,1).")
print("Lex-min chose (6,2) because (6,2) < (7,1) lexicographically.")
print("After visiting (6,2) -> (8,) -> (4,4), all neighbors of (7,1) were exhausted.")


Experiment: Exp 6: SM LexMin start=(1,...,1)

n=1 (OK):
  (1,)

n=2 (OK):
  (1, 1)
  (2,)

n=3 (OK):
  (1, 1, 1)
  (2, 1)
  (3,)

n=4 (OK):
  (1, 1, 1, 1)
  (2, 1, 1)
  (2, 2)
  (4,)
  (3, 1)

n=5 (OK):
  (1, 1, 1, 1, 1)
  (2, 1, 1, 1)
  (2, 2, 1)
  (3, 2)
  (3, 1, 1)
  (4, 1)
  (5,)

n=6 (OK):
  (1, 1, 1, 1, 1, 1)
  (2, 1, 1, 1, 1)
  (2, 2, 1, 1)
  (2, 2, 2)
  (4, 2)
  (3, 2, 1)
  (3, 1, 1, 1)
  (4, 1, 1)
  (5, 1)
  (6,)
  (3, 3)

n=7 (OK):
  (1, 1, 1, 1, 1, 1, 1)
  (2, 1, 1, 1, 1, 1)
  (2, 2, 1, 1, 1)
  (2, 2, 2, 1)
  (3, 2, 2)
  (3, 2, 1, 1)
  (3, 1, 1, 1, 1)
  (4, 1, 1, 1)
  (4, 2, 1)
  (4, 3)
  (3, 3, 1)
  (6, 1)
  (5, 1, 1)
  (5, 2)
  (7,)

n=8 (FAIL): visited 21/22
  Missed:        [(7, 1)]
  Last visited:  (4, 4)
  Its neighbors: [(4, 2, 2), (4, 3, 1), (8,)]

n=9 (FAIL): visited 25/30
  Missed:        [(5, 3, 1), (7, 1, 1), (7, 2), (8, 1), (9,)]
  Last visited:  (3, 3, 3)
  Its neighbors: [(3, 3, 2, 1), (6, 3)]

n=10 (FAIL): visited 41/42
  Missed:        [(9, 1)]
  Last visit

## Experiment 7: Split/Merge, Lex-Max, Start = (1,...,1)

In [14]:
# Experiment 7: Split/merge, start=(1,...,1), priority=lex max.
# Works for n=1–5 and n=7, but fails at n=6 (missing (2,2,1,1) and (2,2,2)).
# Lex-max aggressively merges toward large parts early, skipping over
# small even-shaped partitions that become unreachable.
analyzeExperiment(
  "Exp 7: SM LexMax start=(1,...,1)",
  lambda n: greedyPartition(n, (1,)*n, neighborsSplitMerge, lambda p: tuple(-x for x in p)),
  nMax=12, listingMax=7
)


Experiment: Exp 7: SM LexMax start=(1,...,1)

n=1 (OK):
  (1,)

n=2 (OK):
  (1, 1)
  (2,)

n=3 (OK):
  (1, 1, 1)
  (2, 1)
  (3,)

n=4 (OK):
  (1, 1, 1, 1)
  (2, 1, 1)
  (3, 1)
  (4,)
  (2, 2)

n=5 (OK):
  (1, 1, 1, 1, 1)
  (2, 1, 1, 1)
  (3, 1, 1)
  (4, 1)
  (5,)
  (3, 2)
  (2, 2, 1)

n=6 (FAIL):
  (1, 1, 1, 1, 1, 1)
  (2, 1, 1, 1, 1)
  (3, 1, 1, 1)
  (4, 1, 1)
  (5, 1)
  (6,)
  (4, 2)
  (3, 2, 1)
  (3, 3)
  MISSED: [(2, 2, 1, 1), (2, 2, 2)]

n=7 (OK):
  (1, 1, 1, 1, 1, 1, 1)
  (2, 1, 1, 1, 1, 1)
  (3, 1, 1, 1, 1)
  (4, 1, 1, 1)
  (5, 1, 1)
  (6, 1)
  (7,)
  (5, 2)
  (4, 2, 1)
  (4, 3)
  (3, 3, 1)
  (3, 2, 1, 1)
  (3, 2, 2)
  (2, 2, 2, 1)
  (2, 2, 1, 1, 1)

n=8 (FAIL): visited 21/22
  Missed:        [(2, 2, 1, 1, 1, 1)]
  Last visited:  (2, 2, 2, 2)
  Its neighbors: [(2, 2, 2, 1, 1), (4, 2, 2)]

n=9 (FAIL): visited 23/30
  Missed:        [(2, 2, 1, 1, 1, 1, 1), (2, 2, 2, 1, 1, 1), (2, 2, 2, 2, 1), (3, 2, 1, 1, 1, 1), (3, 2, 2, 1, 1), (3, 2, 2, 2), (3, 3, 1, 1, 1)]
  Last visited:  (3,

## Experiment 8: Split/Merge, Fewest Parts First, Start = (1,...,1)

In [15]:
# Experiment 8: Split/merge, start=(1,...,1), priority=fewest parts (then lex min).
# Works for n=1–7, fails at n=8 missing (3,1,1,1,1,1).
# Prioritizing fewer parts means preferring merges, pushing toward
# consolidated partitions — similar trajectory to Exp 6 but with a different trap.
analyzeExperiment(
  "Exp 8: SM FewestParts start=(1,...,1)",
  lambda n: greedyPartition(n, (1,)*n, neighborsSplitMerge, lambda p: (len(p), p)),
  nMax=12, listingMax=7
)


Experiment: Exp 8: SM FewestParts start=(1,...,1)

n=1 (OK):
  (1,)

n=2 (OK):
  (1, 1)
  (2,)

n=3 (OK):
  (1, 1, 1)
  (2, 1)
  (3,)

n=4 (OK):
  (1, 1, 1, 1)
  (2, 1, 1)
  (2, 2)
  (4,)
  (3, 1)

n=5 (OK):
  (1, 1, 1, 1, 1)
  (2, 1, 1, 1)
  (2, 2, 1)
  (3, 2)
  (5,)
  (4, 1)
  (3, 1, 1)

n=6 (OK):
  (1, 1, 1, 1, 1, 1)
  (2, 1, 1, 1, 1)
  (2, 2, 1, 1)
  (2, 2, 2)
  (4, 2)
  (6,)
  (3, 3)
  (3, 2, 1)
  (5, 1)
  (4, 1, 1)
  (3, 1, 1, 1)

n=7 (OK):
  (1, 1, 1, 1, 1, 1, 1)
  (2, 1, 1, 1, 1, 1)
  (2, 2, 1, 1, 1)
  (2, 2, 2, 1)
  (3, 2, 2)
  (4, 3)
  (7,)
  (5, 2)
  (4, 2, 1)
  (6, 1)
  (3, 3, 1)
  (3, 2, 1, 1)
  (5, 1, 1)
  (4, 1, 1, 1)
  (3, 1, 1, 1, 1)

n=8 (FAIL): visited 21/22
  Missed:        [(3, 1, 1, 1, 1, 1)]
  Last visited:  (5, 1, 1, 1)
  Its neighbors: [(3, 2, 1, 1, 1), (4, 1, 1, 1, 1), (5, 2, 1), (6, 1, 1)]

n=9 (FAIL): visited 29/30
  Missed:        [(3, 1, 1, 1, 1, 1, 1)]
  Last visited:  (5, 1, 1, 1, 1)
  Its neighbors: [(3, 2, 1, 1, 1, 1), (4, 1, 1, 1, 1, 1), (5, 2, 1, 1)

## Experiment 9: Split/Merge, Most Parts First, Start = (1,...,1)

In [16]:
# Experiment 9: Split/merge, start=(1,...,1), priority=most parts (then lex min).
# Works for n=1–7, fails at n=8 missing (7,1) — same as Experiment 6.
# Prioritizing more parts means preferring splits, but the lex-min tiebreaker
# creates the same structural trap as Exp 6 at n=8.
analyzeExperiment(
  "Exp 9: SM MostParts start=(1,...,1)",
  lambda n: greedyPartition(n, (1,)*n, neighborsSplitMerge, lambda p: (-len(p), p)),
  nMax=12, listingMax=7
)


Experiment: Exp 9: SM MostParts start=(1,...,1)

n=1 (OK):
  (1,)

n=2 (OK):
  (1, 1)
  (2,)

n=3 (OK):
  (1, 1, 1)
  (2, 1)
  (3,)

n=4 (OK):
  (1, 1, 1, 1)
  (2, 1, 1)
  (2, 2)
  (4,)
  (3, 1)

n=5 (OK):
  (1, 1, 1, 1, 1)
  (2, 1, 1, 1)
  (2, 2, 1)
  (3, 2)
  (3, 1, 1)
  (4, 1)
  (5,)

n=6 (OK):
  (1, 1, 1, 1, 1, 1)
  (2, 1, 1, 1, 1)
  (2, 2, 1, 1)
  (2, 2, 2)
  (4, 2)
  (3, 2, 1)
  (3, 1, 1, 1)
  (4, 1, 1)
  (5, 1)
  (6,)
  (3, 3)

n=7 (OK):
  (1, 1, 1, 1, 1, 1, 1)
  (2, 1, 1, 1, 1, 1)
  (2, 2, 1, 1, 1)
  (2, 2, 2, 1)
  (3, 2, 2)
  (3, 2, 1, 1)
  (3, 1, 1, 1, 1)
  (4, 1, 1, 1)
  (4, 2, 1)
  (4, 3)
  (3, 3, 1)
  (6, 1)
  (5, 1, 1)
  (5, 2)
  (7,)

n=8 (FAIL): visited 21/22
  Missed:        [(7, 1)]
  Last visited:  (4, 4)
  Its neighbors: [(4, 2, 2), (4, 3, 1), (8,)]

n=9 (FAIL): visited 25/30
  Missed:        [(5, 3, 1), (7, 1, 1), (7, 2), (8, 1), (9,)]
  Last visited:  (3, 3, 3)
  Its neighbors: [(3, 3, 2, 1), (6, 3)]

n=10 (FAIL): visited 41/42
  Missed:        [(9, 1)]
  Last vi

## Summary of Experiments 4–9

| Exp | Start | Priority | First fail | Notes |
|-----|-------|----------|------------|-------|
| 4 | `(n,)` | lex-min | n=4 | Immediately splits into equal halves, cuts off `(k, n-k)` partitions with k≠n/2 |
| 5 | `(n,)` | lex-max | n=4 | Works for odd n=3,5,7; fails inconsistently |
| **6** | `(1,...,1)` | **lex-min** | **n=8** | **Strongest: works n=1–7, misses only `(7,1)` at n=8** |
| 7 | `(1,...,1)` | lex-max | n=6 | Fails earlier than Exp 6; misses small even-shaped partitions |
| 8 | `(1,...,1)` | fewest-parts | n=8 | Same range as Exp 6, different single missed partition |
| 9 | `(1,...,1)` | most-parts | n=8 | Same range as Exp 6, misses same `(7,1)` |

**Conclusion:** Starting from `(1,...,1)` consistently outperforms starting from `(n,)`. Among all experiments, **Experiment 6** is the most promising candidate for a Gray code on integer partitions and merits further investigation to determine whether a modified priority rule can fix the failure at n=8.


# Iteration Toward a Working Algorithm

This section documents the iterative search for a greedy Gray code that works for all n.

**Search strategy:** We fix the split/merge operation and iterate on (a) starting partition and (b) priority rule.

**Key breakthrough:** Combining two neighborhood types — split/merge AND move-one-unit — with Warnsdorff's priority rule produces a Gray code that works for all tested values of n (verified n=1 through n=30).

**Warnsdorff's rule:** At each step, among all unvisited neighbors, choose the one with the *fewest* unvisited neighbors of its own. This classic heuristic (originally for knight's tours) avoids stranding low-degree nodes.


## Combined Neighborhood: Split/Merge + Move-One-Unit

In [17]:
# Move-one-unit: move 1 unit from one part to another.
# (Defined earlier — repeated here for clarity in this section.)
def neighborsMove(partition):
  parts = list(partition)
  results = set()
  n = sum(parts)
  for i in range(len(parts)):
    for j in range(len(parts)):
      if i == j:
        continue
      new_parts = parts[:]
      new_parts[i] -= 1
      new_parts[j] += 1
      new_parts = [p for p in new_parts if p > 0]
      new_parts_sorted = tuple(sorted(new_parts, reverse=True))
      if new_parts_sorted and sum(new_parts_sorted) == n:
        results.add(new_parts_sorted)
    if parts[i] > 1:
      new_parts = parts[:]
      new_parts[i] -= 1
      new_parts.append(1)
      results.add(tuple(sorted(new_parts, reverse=True)))
  return results - {partition}

# Combined neighborhood: union of split/merge and move-one-unit neighbors.
def neighborsCombined(partition):
  return neighborsSplitMerge(partition) | neighborsMove(partition)


## The Winning Algorithm: Warnsdorff's Rule on the Combined Neighborhood

In [18]:
# Warnsdorff's rule on the combined neighborhood, starting from (1,...,1).
#
# At each step, choose the unvisited neighbor with the FEWEST unvisited neighbors
# of its own. Break ties using lexicographic order.
#
# Intuition: visiting low-degree nodes first prevents them from becoming
# stranded (unreachable) later in the traversal.
def greedyWarnsdorff(n):
  visited = set()
  word = (1,) * n
  visited.add(word)
  yield word
  while True:
    candidates = [c for c in neighborsCombined(word) if c not in visited]
    if not candidates:
      break
    word = min(candidates, key=lambda c: (
        len([x for x in neighborsCombined(c) if x not in visited]),
        c
    ))
    visited.add(word)
    yield word


## Results: Warnsdorff's Rule

In [19]:
# Show full listings for small n.
for n in range(1, 9):
  objs = list(greedyWarnsdorff(n))
  print(f"n={n} ({len(objs)} partitions):")
  for i, p in enumerate(objs):
    print(f"  {i+1:2}. {p}")
  print()


n=1 (1 partitions):
   1. (1,)

n=2 (2 partitions):
   1. (1, 1)
   2. (2,)

n=3 (3 partitions):
   1. (1, 1, 1)
   2. (2, 1)
   3. (3,)

n=4 (5 partitions):
   1. (1, 1, 1, 1)
   2. (2, 1, 1)
   3. (2, 2)
   4. (3, 1)
   5. (4,)

n=5 (7 partitions):
   1. (1, 1, 1, 1, 1)
   2. (2, 1, 1, 1)
   3. (2, 2, 1)
   4. (3, 1, 1)
   5. (3, 2)
   6. (4, 1)
   7. (5,)

n=6 (11 partitions):
   1. (1, 1, 1, 1, 1, 1)
   2. (2, 1, 1, 1, 1)
   3. (3, 1, 1, 1)
   4. (2, 2, 1, 1)
   5. (2, 2, 2)
   6. (3, 2, 1)
   7. (3, 3)
   8. (6,)
   9. (4, 2)
  10. (4, 1, 1)
  11. (5, 1)

n=7 (15 partitions):
   1. (1, 1, 1, 1, 1, 1, 1)
   2. (2, 1, 1, 1, 1, 1)
   3. (3, 1, 1, 1, 1)
   4. (2, 2, 1, 1, 1)
   5. (2, 2, 2, 1)
   6. (3, 2, 1, 1)
   7. (4, 1, 1, 1)
   8. (5, 1, 1)
   9. (6, 1)
  10. (7,)
  11. (5, 2)
  12. (3, 2, 2)
  13. (3, 3, 1)
  14. (4, 2, 1)
  15. (4, 3)

n=8 (22 partitions):
   1. (1, 1, 1, 1, 1, 1, 1, 1)
   2. (2, 1, 1, 1, 1, 1, 1)
   3. (3, 1, 1, 1, 1, 1)
   4. (2, 2, 1, 1, 1, 1)
   5. (4, 1, 

In [20]:
# Show which operation connects each pair for n=6.
print("Operation type for each step, n=6:")
objs6 = list(greedyWarnsdorff(6))
for i in range(len(objs6) - 1):
  a, b = objs6[i], objs6[i+1]
  in_sm = b in neighborsSplitMerge(a)
  in_mv = b in neighborsMove(a)
  ops = []
  if in_sm: ops.append("split/merge")
  if in_mv: ops.append("move-1")
  print(f"  {a} -> {b}  [{', '.join(ops)}]")


Operation type for each step, n=6:
  (1, 1, 1, 1, 1, 1) -> (2, 1, 1, 1, 1)  [split/merge, move-1]
  (2, 1, 1, 1, 1) -> (3, 1, 1, 1)  [split/merge, move-1]
  (3, 1, 1, 1) -> (2, 2, 1, 1)  [move-1]
  (2, 2, 1, 1) -> (2, 2, 2)  [split/merge, move-1]
  (2, 2, 2) -> (3, 2, 1)  [move-1]
  (3, 2, 1) -> (3, 3)  [split/merge, move-1]
  (3, 3) -> (6,)  [split/merge]
  (6,) -> (4, 2)  [split/merge]
  (4, 2) -> (4, 1, 1)  [split/merge, move-1]
  (4, 1, 1) -> (5, 1)  [split/merge, move-1]


In [21]:
# Verify for n=1 through n=30.
print("Verification of greedyWarnsdorff:")
nMin, nMax = 1, 30
results = []
for n in range(nMin, nMax + 1):
  allObjects = allPartitions(n)
  goal = len(allObjects)
  objects = list(greedyWarnsdorff(n))
  numObjects = len(objects)
  result = (set(objects) == set(allObjects))
  results.append(result)
  print(f"n={n:2d}: {numObjects:5d}/{goal:5d}  {'OK' if result else 'FAIL'}")

print("\nSummary of greedyWarnsdorff:")
for i, result in enumerate(results):
  print(f"n={i+nMin}: {result}")


Verification of greedyWarnsdorff:
n= 1:     1/    1  OK
n= 2:     2/    2  OK
n= 3:     3/    3  OK
n= 4:     5/    5  OK
n= 5:     7/    7  OK
n= 6:    11/   11  OK
n= 7:    15/   15  OK
n= 8:    22/   22  OK
n= 9:    30/   30  OK
n=10:    42/   42  OK
n=11:    56/   56  OK
n=12:    77/   77  OK
n=13:   101/  101  OK
n=14:   135/  135  OK
n=15:   176/  176  OK
n=16:   231/  231  OK
n=17:   297/  297  OK
n=18:   385/  385  OK
n=19:   490/  490  OK
n=20:   627/  627  OK
n=21:   792/  792  OK
n=22:  1002/ 1002  OK
n=23:  1255/ 1255  OK
n=24:  1575/ 1575  OK
n=25:  1958/ 1958  OK
n=26:  2436/ 2436  OK
n=27:  3010/ 3010  OK
n=28:  3718/ 3718  OK
n=29:  4565/ 4565  OK
n=30:  5604/ 5604  OK

Summary of greedyWarnsdorff:
n=1: True
n=2: True
n=3: True
n=4: True
n=5: True
n=6: True
n=7: True
n=8: True
n=9: True
n=10: True
n=11: True
n=12: True
n=13: True
n=14: True
n=15: True
n=16: True
n=17: True
n=18: True
n=19: True
n=20: True
n=21: True
n=22: True
n=23: True
n=24: True
n=25: True
n=26: True

In [30]:
n = 10
seen = set()
for p in greedyWarnsdorff(n):
    if p in seen:
        print("Already seen:", p)
    else:
        print("New partition:", p)
        seen.add(p)

print(len(seen), "partitions generated.")

New partition: (1, 1, 1, 1, 1, 1, 1, 1, 1, 1)
New partition: (2, 1, 1, 1, 1, 1, 1, 1, 1)
New partition: (3, 1, 1, 1, 1, 1, 1, 1)
New partition: (2, 2, 1, 1, 1, 1, 1, 1)
New partition: (4, 1, 1, 1, 1, 1, 1)
New partition: (5, 1, 1, 1, 1, 1)
New partition: (3, 2, 1, 1, 1, 1, 1)
New partition: (2, 2, 2, 1, 1, 1, 1)
New partition: (2, 2, 2, 2, 1, 1)
New partition: (2, 2, 2, 2, 2)
New partition: (3, 2, 2, 2, 1)
New partition: (3, 2, 2, 1, 1, 1)
New partition: (3, 3, 1, 1, 1, 1)
New partition: (4, 2, 1, 1, 1, 1)
New partition: (6, 1, 1, 1, 1)
New partition: (7, 1, 1, 1)
New partition: (5, 2, 1, 1, 1)
New partition: (4, 3, 1, 1, 1)
New partition: (3, 3, 2, 1, 1)
New partition: (3, 3, 3, 1)
New partition: (3, 3, 2, 2)
New partition: (4, 2, 2, 2)
New partition: (4, 2, 2, 1, 1)
New partition: (4, 4, 1, 1)
New partition: (8, 1, 1)
New partition: (6, 2, 1, 1)
New partition: (5, 3, 1, 1)
New partition: (5, 2, 2, 1)
New partition: (6, 2, 2)
New partition: (7, 2, 1)
New partition: (9, 1)
New partitio

## Summary of Iteration

The path to the working algorithm:

1. **Experiment 6** (split/merge, lex-min, start `(1,...,1)`) — worked up to n=7, failed at n=8 missing only `(7,1)`.
2. **Root cause:** At `(6,1,1)`, lex-min chose `(6,2)` over `(7,1)` because `(6,2) < (7,1)` lexicographically. After visiting `(6,2)`, `(8,)`, and `(4,4)`, all three neighbors of `(7,1)` were exhausted.
3. **Warnsdorff's rule (split/merge only):** Choosing the neighbor with fewest future options still failed at n=8 (same trap).
4. **Combined neighborhood (split/merge + move-one-unit) with lex-min:** Worked for n=1–9 and most of n=11–20, but failed at n=10 (missed 8 partitions).
5. **Combined neighborhood + Warnsdorff:** Works for all tested n up to n=30. ✓

**The working algorithm:**
- **Objects:** integer partitions of n (tuples in weakly decreasing order)
- **Start:** `(1, 1, ..., 1)`
- **Operations:** split one part into two, OR merge two parts into one, OR move one unit between parts
- **Priority:** fewest unvisited neighbors first (Warnsdorff); break ties by lexicographic order
